In [13]:
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms, datasets
import torch

class preprocessing_ours:
    def __init__(self, size, crop):
        self.size = size
        self.crop = crop

    def __call__(self, img):
        # 1. PIL → NumPy & Crop
        img_np = np.array(img.convert("RGB"))
        if self.crop > 0:
            h, w, _ = img_np.shape
            img_np = img_np[self.crop:h - self.crop, self.crop:w - self.crop, :]

        # 2. Grayscale
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        # 3. Edge extraction (LoG + Sobel)
        log_edge = np.abs(cv2.Laplacian(gray, cv2.CV_64F, ksize=3)) 
        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_edge = np.sqrt(sobelx ** 2 + sobely ** 2)

        # Normalize
        log_norm = cv2.normalize(log_edge, None, 0, 1.0, cv2.NORM_MINMAX)
        sobel_norm = cv2.normalize(sobel_edge, None, 0, 1.0, cv2.NORM_MINMAX)

        # Weighted combination
        log_weight = np.mean(log_norm)
        sobel_weight = np.mean(sobel_norm)
        total = log_weight + sobel_weight
        w1 = log_weight / total
        w2 = sobel_weight / total
        edge = w1 * log_norm + w2 * sobel_norm
        edge = cv2.normalize(edge, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        # 4. Blur & Resize
        blurred = cv2.GaussianBlur(edge, (3, 3), 1)
        resized = cv2.resize(blurred, (self.size, self.size), interpolation=cv2.INTER_CUBIC)

        # 5. Binarize
        _, binarized = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # 6. To Tensor
        final_img = Image.fromarray(binarized)
        return transforms.functional.to_tensor(final_img)  # (1, H, W)

class preprocessing_baseline:
    def __init__(self, size, crop):
        self.size = size
        self.crop = crop
        
    def __call__(self, img):
        img_np = np.array(img.convert("RGB"))
        if self.crop > 0:
            h, w, _ = img_np.shape
            img_np = img_np[self.crop:h - self.crop, self.crop:w - self.crop, :]

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        blurred = cv2.GaussianBlur(gray, (3, 3), 1)
        resized = cv2.resize(blurred, (self.size, self.size), interpolation=cv2.INTER_CUBIC)
        _, binarized = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        final_img = Image.fromarray(binarized)
        return transforms.functional.to_tensor(final_img)  # (1, H, W)

class TwoChannelPreprocessing:
    def __init__(self, size, crop):
        self.ours = preprocessing_ours(size, crop)
        self.base = preprocessing_baseline(size, crop)

    def __call__(self, img):
        base_tensor = self.base(img)  # (1, H, W)
        ours_tensor = self.ours(img)  # (1, H, W)
        stacked = torch.cat([base_tensor, ours_tensor], dim=0)  # (2, H, W)
        return stacked

In [14]:
import torch
two_channel_transform = transforms.Compose([
    TwoChannelPreprocessing(size=45, crop=2)
])
trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train', transform=two_channel_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=40, shuffle=True, num_workers=2, drop_last=True)

testset  = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test',  transform=two_channel_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size = 40, shuffle=False, num_workers=2, drop_last=True)

print(f"# of trainset = {len(trainset)}")
print(f"# of trainloader = {len(trainloader)}")
target, label = trainset[5000]
print(f"input size: {target.shape}, {label}")     

# of trainset = 8000
# of trainloader = 200
input size: torch.Size([2, 45, 45]), 2


In [15]:
import os
import numpy as np 
import torch.optim as optim   
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs, save_path='./prob2_3_1_weight2'):
    os.makedirs(save_path, exist_ok=True)  
    history = []

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss, correct, total = 0, 0, 0

        for X, y in trainloader: 
            X = X.to(device);y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total
        history.append((epoch + 1, avg_loss, avg_accuracy))
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")

        if (epoch + 1) % 10 == 0:
            filename = os.path.join(save_path, f"weight{epoch+1}.pth")
            torch.save(model.state_dict(), filename)
            print(f"Saved checkpoint: {filename}")

    return np.array(history)
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [16]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
prob2_3 = models.resnet18()
num_ftrs = prob2_3.fc.in_features
prob2_3.conv1 = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3, bias=False)
prob2_3.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)

prob2_3 = prob2_3.to(device)

In [17]:
# 하이퍼파라미터 설정
num_epochs = 50
lr = 0.001
optimizer = torch.optim.Adam(prob2_3.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

# 학습 실행
history = train(prob2_3, device, trainloader, optimizer, criterion, num_epochs)

  2%|▉                                           | 1/50 [00:20<16:40, 20.42s/it]

Epoch [1/50] - Loss: 0.938739, Accuracy: 0.5843


  4%|█▊                                          | 2/50 [00:40<16:12, 20.26s/it]

Epoch [2/50] - Loss: 0.739141, Accuracy: 0.6985


  6%|██▋                                         | 3/50 [01:01<15:56, 20.35s/it]

...

Epoch [48/50] - Loss: 0.020549, Accuracy: 0.9936


 98%|██████████████████████████████████████████▏| 49/50 [16:38<00:19, 19.49s/it]

Epoch [49/50] - Loss: 0.013327, Accuracy: 0.9960


100%|███████████████████████████████████████████| 50/50 [16:58<00:00, 20.36s/it]

Epoch [50/50] - Loss: 0.015083, Accuracy: 0.9945
Saved checkpoint: ./prob2_3_1_weight2/weight50.pth


In [18]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
model_paths = [
    './prob2_3_1_weight/weight10.pth',
    './prob2_3_1_weight/weight20.pth',
    './prob2_3_1_weight/weight30.pth',
    './prob2_3_1_weight/weight40.pth',
    './prob2_3_1_weight/weight50.pth',
]
criterion = torch.nn.CrossEntropyLoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

for path in model_paths:
    model = models.resnet18()
    model.conv1 = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 4)
    )
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)
    test(model, device, testloader, criterion)

100%|███████████████████████████████████████████| 19/19 [00:02<00:00,  9.31it/s]


test loss : 1.2659 / test_accuracy : 0.6697


100%|███████████████████████████████████████████| 19/19 [00:01<00:00,  9.50it/s]


test loss : 1.5994 / test_accuracy : 0.7039


100%|███████████████████████████████████████████| 19/19 [00:02<00:00,  9.42it/s]


test loss : 2.0341 / test_accuracy : 0.6526


100%|███████████████████████████████████████████| 19/19 [00:02<00:00,  9.37it/s]


test loss : 1.8695 / test_accuracy : 0.6947


100%|███████████████████████████████████████████| 19/19 [00:02<00:00,  9.47it/s]

test loss : 2.0545 / test_accuracy : 0.6539
